# Pratica — Lezione 6. Controllo senza modello: SARSA e Q-learning

Questa pratica è tutta tua. Lavorerai in autonomia sullo stesso ambiente **Cliff Walking** (la scogliera) che hai costruito in classe e risolverai due esercizi sull'aggiornamento di **SARSA** (on-policy) e di **Q-learning** (off-policy).

**Come lavorare con questo notebook:**
1. Esegui prima la sezione "Preparazione", che ricostruisce l'ambiente della lezione
2. In ogni esercizio, completa le righe contrassegnate con `# TODO`
3. Esegui la tua soluzione e confronta il risultato con quanto descritto nella consegna
4. Solo dopo, apri la soluzione nascosta per verificare la tua risposta

> **Non guardare la soluzione prima di averci provato.** Il valore di questo esercizio sta nel confrontarsi un po' con la differenza tra "il Q dell'azione che seguo" e "il massimo Q possibile", non nel leggere la risposta.

## Preparazione: recupera l'ambiente della lezione

Questa cella riproduce esattamente l'ambiente della scogliera e le funzioni ausiliarie (`paso`, `politica_epsilon_greedy`, `extraer_ruta_greedy`, `media_movil`, `dibujar_ruta`) che hai costruito nel notebook teorico. Gli esercizi di oggi riutilizzano questi stessi nomi, quindi non cambiarli.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FILAS, COLUMNAS = 4, 12
INICIO = (3, 0)
META = (3, 11)
ACANTILADO = [(3, c) for c in range(1, 11)]
NUM_ACCIONES = 4


def paso(estado, accion):
    """Aplica una acción y devuelve (nuevo_estado, recompensa, terminado)."""
    fila, col = estado
    if accion == 0:
        fila = max(fila - 1, 0)
    elif accion == 1:
        fila = min(fila + 1, FILAS - 1)
    elif accion == 2:
        col = max(col - 1, 0)
    elif accion == 3:
        col = min(col + 1, COLUMNAS - 1)

    nuevo_estado = (fila, col)

    if nuevo_estado in ACANTILADO:
        return INICIO, -100, False
    if nuevo_estado == META:
        return nuevo_estado, -1, True
    return nuevo_estado, -1, False


def politica_epsilon_greedy(Q, estado, epsilon):
    """Con probabilidad epsilon explora; si no, elige la mejor acción conocida."""
    if np.random.rand() < epsilon:
        return np.random.randint(NUM_ACCIONES)
    valores = Q[estado]
    return np.random.choice(np.flatnonzero(valores == valores.max()))


def extraer_ruta_greedy(Q, max_pasos=50):
    """Sigue la política puramente codiciosa (sin exploración) desde el inicio."""
    estado = INICIO
    ruta = [estado]
    for _ in range(max_pasos):
        if estado == META:
            break
        accion = np.argmax(Q[estado])
        estado, _, terminado = paso(estado, accion)
        ruta.append(estado)
        if terminado:
            break
    return ruta


def media_movil(valores, ventana=20):
    return np.convolve(valores, np.ones(ventana) / ventana, mode="valid")


def dibujar_ruta(ax, ruta, titulo):
    cuadricula = np.zeros((FILAS, COLUMNAS))
    for (f, c) in ACANTILADO:
        cuadricula[f, c] = -1
    ax.imshow(cuadricula, cmap="Pastel1")
    filas_ruta = [p[0] for p in ruta]
    columnas_ruta = [p[1] for p in ruta]
    ax.plot(columnas_ruta, filas_ruta, marker="o", color="tab:blue")
    ax.scatter(*INICIO[::-1], color="green", s=100, label="Inicio", zorder=5)
    ax.scatter(*META[::-1], color="red", s=100, label="Meta", zorder=5)
    ax.set_title(titulo)
    ax.invert_yaxis()
    ax.legend(loc="upper center")


---

## Esercizio 1: completa l'aggiornamento di SARSA (on-policy)

SARSA sceglie l'azione successiva **prima** di aggiornare, con la stessa politica ε-greedy che usa per agire, e usa il Q di quell'azione specifica come obiettivo. Completa la funzione `entrenar_sarsa_ejercicio` nei tre spazi vuoti che mancano all'interno del ciclo:

```
Q(S,A) si avvicina un po' a: ricompensa ricevuta + valore scontato di Q nello stato
                             successivo, per l'azione che la politica ha scelto davvero
```

**Suggerimenti:**
- Il primo spazio sceglie `siguiente_accion` con `politica_epsilon_greedy(Q, siguiente_estado, epsilon)`, esattamente come si sceglie `accion` all'inizio di ogni episodio
- Il secondo spazio calcola `objetivo`, combinando `recompensa`, `gamma` e `Q[siguiente_estado][siguiente_accion]`
- Il terzo spazio aggiorna `Q[estado][accion]` avvicinandolo a `objetivo` con il tasso di apprendimento `alpha`: è la stessa regola TD che conosci già dalla lezione precedente, `Q[estado][accion] += alpha * (objetivo - Q[estado][accion])`
- Con gli iperparametri di default (500 episodi, α=0.5, γ=1.0, ε=0.1) e seme 0, la ricompensa media degli ultimi 50 episodi dovrebbe aggirarsi intorno a −32

In [ ]:
def entrenar_sarsa_ejercicio(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """SARSA: actualización on-policy. Completa las tres líneas marcadas con TODO."""
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        accion = politica_epsilon_greedy(Q, estado, epsilon)
        recompensa_total = 0
        terminado = False

        while not terminado:
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # TODO: elige la siguiente acción con la política epsilon-greedy.
            # SARSA necesita saber AHORA qué acción va a tomar después.
            siguiente_accion = None

            # TODO: calcula el objetivo on-policy: recompensa + gamma * Q de
            # la acción que SÍ vas a tomar a continuación (siguiente_accion)
            objetivo = None

            # TODO: actualiza Q[estado][accion] acercándolo a 'objetivo'
            # con la tasa de aprendizaje alpha
            Q[estado][accion] += 0

            estado, accion = siguiente_estado, siguiente_accion
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


np.random.seed(0)
Q_sarsa_ejercicio, recompensas_sarsa_ejercicio = entrenar_sarsa_ejercicio()

print(f"Recompensa media de los últimos 50 episodios (SARSA): {np.mean(recompensas_sarsa_ejercicio[-50:]):.1f}")
print("Ruta final aprendida (SARSA):", extraer_ruta_greedy(Q_sarsa_ejercicio))


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def entrenar_sarsa_ejercicio(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        accion = politica_epsilon_greedy(Q, estado, epsilon)
        recompensa_total = 0
        terminado = False

        while not terminado:
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # Scegliamo già la azione successiva con la politica epsilon-greedy:
            # è la azione che eseguiremo davvero nel passo successivo
            siguiente_accion = politica_epsilon_greedy(Q, siguiente_estado, epsilon)

            # Obiettivo on-policy: usiamo il Q di quell'azione specifica,
            # non il massimo tra tutte le azioni possibili
            objetivo = recompensa + gamma * Q[siguiente_estado][siguiente_accion]

            # Regola TD di sempre: avviciniamo Q[estado][accion] all'obiettivo
            Q[estado][accion] += alpha * (objetivo - Q[estado][accion])

            estado, accion = siguiente_estado, siguiente_accion
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


np.random.seed(0)
Q_sarsa_ejercicio, recompensas_sarsa_ejercicio = entrenar_sarsa_ejercicio()

print(f"Ricompensa media degli ultimi 50 episodi (SARSA): {np.mean(recompensas_sarsa_ejercicio[-50:]):.1f}")
print("Percorso finale appreso (SARSA):", extraer_ruta_greedy(Q_sarsa_ejercicio))

# Output atteso approssimativo (i valori esatti possono variare un po'
# a seconda della versione di numpy, ma dovrebbero avvicinarsi a questi):
# Ricompensa media degli ultimi 50 episodi (SARSA): -32.5
# Percorso finale appreso (SARSA): [(3, 0), (2, 0), (1, 0), (1, 1), (0, 1), ...,
#                                 (0, 11), (1, 11), (2, 11), (3, 11)]
#
# Osserva il percorso: si allontana dalla scogliera salendo verso la riga 0
# (la più lontana dal bordo), proprio il comportamento prudente che ti
# aspettavi da un algoritmo on-policy.
```
</details>

---

## Esercizio 2: completa l'aggiornamento di Q-learning (off-policy) e confronta la stabilizzazione

Q-learning cambia un solo dettaglio rispetto a SARSA: invece del Q dell'azione che prenderai davvero dopo, usa il **massimo** Q possibile tra tutte le azioni dello stato successivo. Completa `entrenar_q_learning_ejercicio` e poi usa il codice già fornito per confrontare quanti episodi impiega ciascun algoritmo a stabilizzare la propria ricompensa.

```
Q(S,A) si avvicina un po' a: ricompensa ricevuta + valore scontato del miglior Q
                             possibile nello stato successivo (il massimo su tutte le azioni)
```

**Suggerimenti:**
- Il primo spazio calcola `mejor_valor_siguiente` con `Q[siguiente_estado].max()`, senza scegliere nessuna azione specifica
- Il secondo spazio calcola `objetivo`, combinando `recompensa`, `gamma` e `mejor_valor_siguiente`
- Il terzo spazio è identico a quello di SARSA: `Q[estado][accion] += alpha * (objetivo - Q[estado][accion])`
- La funzione `episodio_de_estabilizacion` è già completa: cerca il primo episodio in cui la media mobile della ricompensa si avvicina a meno di un margine del suo valore finale. È un modo approssimativo per capire quando la curva smette di migliorare ed entra nel suo plateau
- Esegui questa cella **dopo** quella dell'esercizio 1, senza rieseguire quella cella: così riutilizzi `recompensas_sarsa_ejercicio` per il confronto finale

In [ ]:
def entrenar_q_learning_ejercicio(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """Q-learning: actualización off-policy con el máximo. Completa los tres TODO."""
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        recompensa_total = 0
        terminado = False

        while not terminado:
            accion = politica_epsilon_greedy(Q, estado, epsilon)
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # TODO: calcula el mejor Q posible en el estado siguiente,
            # sin importar qué acción se elija después en la práctica
            mejor_valor_siguiente = None

            # TODO: calcula el objetivo off-policy: recompensa + gamma * mejor_valor_siguiente
            objetivo = None

            # TODO: actualiza Q[estado][accion] acercándolo a 'objetivo'
            # con la tasa de aprendizaje alpha (la misma regla TD de siempre)
            Q[estado][accion] += 0

            estado = siguiente_estado
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


def episodio_de_estabilizacion(recompensas, ventana=20, margen=10.0):
    """Heurística sencilla: primer episodio en el que la media móvil
    (ventana de 20 episodios) queda a menos de 'margen' de su propio
    valor final. Marca, de forma aproximada, cuándo la curva deja de
    mejorar con claridad y entra en su meseta."""
    mm = media_movil(recompensas, ventana)
    objetivo = mm[-1]
    dentro_del_margen = np.abs(mm - objetivo) < margen
    primer_indice = int(np.argmax(dentro_del_margen))
    return primer_indice + ventana


Q_qlearning_ejercicio, recompensas_qlearning_ejercicio = entrenar_q_learning_ejercicio()

print(f"Recompensa media de los últimos 50 episodios (Q-learning): {np.mean(recompensas_qlearning_ejercicio[-50:]):.1f}")

ep_sarsa = episodio_de_estabilizacion(recompensas_sarsa_ejercicio)
ep_qlearning = episodio_de_estabilizacion(recompensas_qlearning_ejercicio)

print(f"\nSARSA se estabiliza aproximadamente en el episodio {ep_sarsa}")
print(f"Q-learning se estabiliza aproximadamente en el episodio {ep_qlearning}")

plt.figure(figsize=(9, 5))
plt.plot(media_movil(recompensas_sarsa_ejercicio), label="SARSA (on-policy)")
plt.plot(media_movil(recompensas_qlearning_ejercicio), label="Q-learning (off-policy)")
plt.axvline(ep_sarsa, color="tab:blue", linestyle="--", alpha=0.5)
plt.axvline(ep_qlearning, color="tab:orange", linestyle="--", alpha=0.5)
plt.xlabel("Episodio")
plt.ylabel("Recompensa por episodio (media móvil)")
plt.title("Comparación de estabilización: SARSA vs Q-learning")
plt.legend()
plt.tight_layout()
plt.show()


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def entrenar_q_learning_ejercicio(num_episodios=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((FILAS, COLUMNAS, NUM_ACCIONES))
    recompensas_por_episodio = []

    for _ in range(num_episodios):
        estado = INICIO
        recompensa_total = 0
        terminado = False

        while not terminado:
            accion = politica_epsilon_greedy(Q, estado, epsilon)
            siguiente_estado, recompensa, terminado = paso(estado, accion)

            # Massimo Q dello stato successivo: non dipende da quale azione
            # si sceglierà poi nella pratica, questa è la chiave off-policy
            mejor_valor_siguiente = Q[siguiente_estado].max()

            # Obiettivo off-policy: ricompensa + valore scontato del miglior
            # Q possibile, supponendo che da ora in poi si agisca in modo ottimale
            objetivo = recompensa + gamma * mejor_valor_siguiente

            # Stessa regola TD di SARSA
            Q[estado][accion] += alpha * (objetivo - Q[estado][accion])

            estado = siguiente_estado
            recompensa_total += recompensa

        recompensas_por_episodio.append(recompensa_total)

    return Q, recompensas_por_episodio


def episodio_de_estabilizacion(recompensas, ventana=20, margen=10.0):
    mm = media_movil(recompensas, ventana)
    objetivo = mm[-1]
    dentro_del_margen = np.abs(mm - objetivo) < margen
    primer_indice = int(np.argmax(dentro_del_margen))
    return primer_indice + ventana


Q_qlearning_ejercicio, recompensas_qlearning_ejercicio = entrenar_q_learning_ejercicio()

print(f"Ricompensa media degli ultimi 50 episodi (Q-learning): {np.mean(recompensas_qlearning_ejercicio[-50:]):.1f}")

ep_sarsa = episodio_de_estabilizacion(recompensas_sarsa_ejercicio)
ep_qlearning = episodio_de_estabilizacion(recompensas_qlearning_ejercicio)

print(f"\nSARSA si stabilizza approssimativamente all'episodio {ep_sarsa}")
print(f"Q-learning si stabilizza approssimativamente all'episodio {ep_qlearning}")

# Output atteso approssimativo, con seme 0 ed eseguendo l'esercizio 1
# subito prima di questo (stesso flusso di numeri casuali):
# Ricompensa media degli ultimi 50 episodi (Q-learning): -38.0
# SARSA si stabilizza approssimativamente all'episodio 75
# Q-learning si stabilizza approssimativamente all'episodio 84
```

**Perché il numero è solo approssimativo:** qui `epsilon` resta fisso a 0.1 per tutto l'addestramento —non decade, quindi non rispetta del tutto la condizione GLIE vista nel riepilogo della lezione—. Questo significa che entrambi gli agenti continuano a esplorare e quindi, fino all'ultimo episodio, ogni tanto cadono ancora dalla scogliera. Il "plateau" non è mai perfettamente piatto: è una fascia rumorosa attorno a un valore medio. Il numero restituito da `episodio_de_estabilizacion` è una fotografia approssimativa di quando quella fascia inizia; per una lettura più completa, guarda sempre anche la forma della curva nel grafico, non solo il numero isolato.
</details>

---

## Per continuare a esercitarti

Se ti è avanzato del tempo, torna alle funzioni complete e prova queste varianti del capitolo:

1. **Cambia epsilon.** Porta `epsilon` a 0.3 nelle due funzioni di addestramento e riesegui. Osserva se il divario tra la ricompensa di SARSA e quella di Q-learning durante l'addestramento si allarga o si riduce
2. **Cambia la penalità della scogliera.** Nella funzione `paso`, sostituisci il −100 con, per esempio, −10 o −500, e confronta come cambiano i due percorsi finali. Annota le tue osservazioni: con una penalità molto bassa, i due percorsi finiscono per assomigliarsi?

**Prossima tappa:** nella Lezione 7 sostituirai la tabella Q con una rete neurale, perché una tabella esplicita smette di essere praticabile non appena il numero di stati esplode. Qui nascono le reti Q profonde (DQN, da *Deep Q-Networks*). Salva questo notebook: la logica dell'aggiornamento off-policy che hai appena programmato ricompare praticamente identica!